In [ ]:
import SimpleITK as sitk

# Dateien
t1_path = "T1.nii"
t2_path = "T2.nii"
out_path = "T2_registered_to_T1.nii"

# Bilder laden
fixed = sitk.ReadImage(t1_path, sitk.sitkFloat32)   # T1 = Referenz
moving = sitk.ReadImage(t2_path, sitk.sitkFloat32)  # T2 = wird transformiert

# Initialer rigid Transform
initial_transform = sitk.CenteredTransformInitializer(
    fixed,
    moving,
    sitk.Euler3DTransform(),
    sitk.CenteredTransformInitializerFilter.GEOMETRY
)

# Registrierung
registration_method = sitk.ImageRegistrationMethod()
registration_method.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
registration_method.SetMetricSamplingStrategy(registration_method.RANDOM)
registration_method.SetMetricSamplingPercentage(0.1)

registration_method.SetInterpolator(sitk.sitkLinear)

registration_method.SetOptimizerAsGradientDescent(
    learningRate=1.0,
    numberOfIterations=100,
    convergenceMinimumValue=1e-6,
    convergenceWindowSize=10
)
registration_method.SetOptimizerScalesFromPhysicalShift()

registration_method.SetInitialTransform(initial_transform, inPlace=False)

# Multi-resolution hilft oft sehr
registration_method.SetShrinkFactorsPerLevel([4, 2, 1])
registration_method.SetSmoothingSigmasPerLevel([2, 1, 0])
registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

final_transform = registration_method.Execute(fixed, moving)

# T2 in T1-Raum resamplen
moving_resampled = sitk.Resample(
    moving,
    fixed,
    final_transform,
    sitk.sitkLinear,
    0.0,
    moving.GetPixelID()
)

# Speichern
sitk.WriteImage(moving_resampled, out_path)

print("Saved:", out_path)